# DCF Valuation — C25 Index Stocks

This notebook builds a **Discounted Cash Flow (DCF)** model for all stocks in the Danish C25 index.  
It fetches live financial data via `yfinance` and estimates each stock's **intrinsic value** compared to its current market price.

---

### The DCF Model — Quick Primer

The core idea: *a company is worth the present value of all the cash it will generate in the future.*

$$
\text{Intrinsic Value} = \sum_{t=1}^{n} \frac{FCF_t}{(1+r)^t} + \frac{\text{Terminal Value}}{(1+r)^n}
$$

Where:
- $FCF_t$ = Free Cash Flow in year $t$
- $r$ = Discount rate (WACC)
- Terminal Value = value beyond the projection window (Gordon Growth Model)


In [ ]:
# Install dependencies if needed
# !pip install yfinance pandas matplotlib

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Parameters
Adjust these to test different scenarios.

In [ ]:
# C25 tickers
C25_STOCKS = {
    'NOVO-B.CO':   'Novo Nordisk',
    "FLS.CO":       "FLSmidth",
    'MAERSK-B.CO': 'A.P. Møller-Mærsk B',
    'ORSTED.CO':   'Ørsted',
    'CARL-B.CO':   'Carlsberg',
    'NSIS-B.CO':   'Novonesis',
    'DSV.CO':      'DSV',
    'DEMANT.CO':   'Demant',
    'GN.CO':       'GN Store Nord',
    'COLO-B.CO':   'Coloplast',
    'AMBU-B.CO':   'Ambu',
    'TRYG.CO':     'Tryg',
    'RBREW.CO':    'Royal Unibrew',
    'ROCK-B.CO':   'Rockwool',
    'PNDORA.CO':   'Pandora',
    'DANSKE.CO':   'Danske Bank',
    'GMAB.CO':     'Genmab',
    'VWS.CO':      'Vestas Wind Systems',
    'ZEAL.CO':     'Zealand Pharma',
    'ISS.CO':      'ISS',
    'NKT.CO':      'NKT',
    'NDA-DK.CO':   'Nordea Bank',
    'JYSK.CO':     'Jyske Bank',
    'SYDB.CO':     'Sydbank',
    'BAVA.CO':     'Bavarian Nordic',
}

# DCF assumptions
PROJECTION_YEARS = 5      # forecast horizon
DISCOUNT_RATE    = 0.10   # WACC
TERMINAL_GROWTH  = 0.025  # long-run growth
FCF_GROWTH_RATE  = 0.07   # near-term FCF growth

## 2. Fetch Data & Run DCF

In [ ]:
def fetch_stock_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        info  = stock.info
        cf    = stock.cashflow
        if cf is None or cf.empty:
            return None
        op_cf_key = next((k for k in cf.index if 'Operating' in k and 'Cash' in k), None)
        capex_key = next((k for k in cf.index if 'Capital' in k and 'Expenditure' in k), None)
        if op_cf_key is None:
            return None
        fcf    = cf.loc[op_cf_key].iloc[0] - abs(cf.loc[capex_key].iloc[0] if capex_key else 0)
        shares = info.get('sharesOutstanding') or info.get('impliedSharesOutstanding')
        price  = info.get('currentPrice') or info.get('regularMarketPrice')
        if not shares or not price or fcf <= 0:
            return None
        return {'ticker': ticker, 'name': info.get('longName', ticker),
                'current_price': price, 'shares': shares, 'fcf': fcf,
                'currency': info.get('currency', 'DKK')}
    except Exception:
        return None

def run_dcf(fcf, shares):
    pvs = [fcf * (1 + FCF_GROWTH_RATE)**y / (1 + DISCOUNT_RATE)**y
           for y in range(1, PROJECTION_YEARS + 1)]
    terminal_fcf = fcf * (1 + FCF_GROWTH_RATE)**PROJECTION_YEARS
    pv_terminal  = (terminal_fcf * (1 + TERMINAL_GROWTH) /
                    (DISCOUNT_RATE - TERMINAL_GROWTH)) / (1 + DISCOUNT_RATE)**PROJECTION_YEARS
    return (sum(pvs) + pv_terminal) / shares

results = []
for ticker, name in C25_STOCKS.items():
    print(f'Fetching {name}...', end=' ')
    d = fetch_stock_data(ticker)
    if d is None:
        print('skipped')
        continue
    intrinsic = run_dcf(d['fcf'], d['shares'])
    margin    = (intrinsic - d['current_price']) / d['current_price'] * 100
    verdict   = '🟢 Undervalued' if margin > 10 else ('🔴 Overvalued' if margin < -10 else '🟡 Fair')
    print(verdict)
    results.append({'Company': name, 'Ticker': ticker, 'Currency': d['currency'],
                    'Current Price': round(d['current_price'], 2),
                    'Intrinsic Value': round(intrinsic, 2),
                    'Margin of Safety (%)': round(margin, 1),
                    'Verdict': verdict})

df = pd.DataFrame(results)

## 3. Results Table

In [ ]:
df.sort_values('Margin of Safety (%)', ascending=False).reset_index(drop=True)

## 4. Visualisation

In [ ]:
df_plot = df.sort_values('Margin of Safety (%)', ascending=True)
colors  = ['#00c49a' if m > 10 else '#ff4d6d' if m < -10 else '#ffd166'
           for m in df_plot['Margin of Safety (%)']]

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bars = ax.barh(df_plot['Company'], df_plot['Margin of Safety (%)'],
               color=colors, edgecolor='none', height=0.6)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

for bar, val in zip(bars, df_plot['Margin of Safety (%)']):
    x = bar.get_width()
    ax.text(x + (1 if x >= 0 else -1), bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center', ha='left' if x >= 0 else 'right',
            color='white', fontsize=8.5)

ax.set_xlabel('Margin of Safety (%)', color='#aaaaaa')
ax.set_title('DCF Valuation — C25 Index\nMargin of Safety vs Current Price',
             color='white', fontsize=14, fontweight='bold', pad=14)
ax.tick_params(colors='#aaaaaa')
for spine in ax.spines.values():
    spine.set_visible(False)

legend_items = [
    mpatches.Patch(color='#00c49a', label='Undervalued (>+10%)'),
    mpatches.Patch(color='#ffd166', label='Fairly Valued (±10%)'),
    mpatches.Patch(color='#ff4d6d', label='Overvalued (<-10%)'),
]
ax.legend(handles=legend_items, loc='lower right', framealpha=0.15,
          labelcolor='white', fontsize=9)
plt.tight_layout()
plt.savefig('output/dcf_c25.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 5. Key Assumptions & Limitations

| Assumption | Value | Note |
|---|---|---|
| Discount rate (WACC) | 10% | Adjust per sector risk profile |
| FCF growth rate | 7% | Uniform — real analysis would use per-company estimates |
| Terminal growth rate | 2.5% | Approx. long-run Danish GDP growth |
| Projection horizon | 5 years | Standard for DCF |

> ⚠️ This model uses uniform growth assumptions for simplicity. A production-grade analysis would use company-specific guidance, analyst consensus estimates, and sector-adjusted discount rates. **This is not investment advice.**